# Conditional Routing in Graphs

Real workflows rarely execute the same steps for every input. Conditional routing lets your graph inspect the current state after a node completes and choose which node to visit next. A routing function returns a string key; that key maps to the next node name.

**What you'll build:** A question-answering graph that classifies the user's question type, then routes to a specialist node — factual lookup, opinion synthesis, or creative generation — based on the classification.

**Time:** ~15 min | **Difficulty:** Beginner

## Prerequisites & Setup

You need an `OPENAI_API_KEY` environment variable set.

**What you'll learn:**
- Write a routing function that returns a string based on state
- Register conditional edges with `add_conditional_edges()`
- Map routing function return values to node names
- Handle the "default" / fallback route

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key before running
# os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 1: Define State and Classification

The state tracks the question, its classified type, and the final answer.

In [ ]:
from __future__ import annotations
import asyncio
from dataclasses import dataclass

from synapsekit.graph import StateGraph
from synapsekit.llms.openai import OpenAILLM
from synapsekit import LLMConfig

@dataclass
class QAState:
    question: str
    question_type: str = ""   # "factual" | "opinion" | "creative" | "unknown"
    answer: str = ""

## Step 2: Implement Nodes

One classifier node determines the question type; four handler nodes produce answers for each type.

In [ ]:
llm = OpenAILLM(model="gpt-4o-mini", config=LLMConfig(temperature=0.0))

async def classify_question(state: QAState) -> QAState:
    """Determine the question type so the router can pick the right handler."""
    response = await llm.agenerate(
        f"""Classify the following question into exactly one category:
- factual: has a definitive correct answer based on facts or data
- opinion: requires synthesizing multiple viewpoints or perspectives
- creative: calls for imagination, storytelling, or open-ended generation
- unknown: does not fit the above categories

Question: {state.question}

Respond with only the category name."""
    )
    state.question_type = response.text.strip().lower()
    print(f"[classify] Type: {state.question_type}")
    return state


async def handle_factual(state: QAState) -> QAState:
    """Answer factual questions concisely and accurately."""
    response = await llm.agenerate(
        f"Answer this factual question accurately and concisely: {state.question}"
    )
    state.answer = response.text
    print(f"[factual] Answered.")
    return state


async def handle_opinion(state: QAState) -> QAState:
    """Synthesize multiple perspectives for opinion questions."""
    response = await llm.agenerate(
        f"This question calls for multiple perspectives. Present at least two "
        f"distinct viewpoints on: {state.question}"
    )
    state.answer = response.text
    print(f"[opinion] Synthesized perspectives.")
    return state


async def handle_creative(state: QAState) -> QAState:
    """Generate a creative, imaginative response."""
    response = await llm.agenerate(
        f"Respond to this with creativity and imagination: {state.question}"
    )
    state.answer = response.text
    print(f"[creative] Generated response.")
    return state


async def handle_unknown(state: QAState) -> QAState:
    """Fallback for questions that don't fit any category."""
    state.answer = (
        "I'm not sure how to categorize that question. "
        "Could you rephrase it or provide more context?"
    )
    print(f"[unknown] Fell through to default handler.")
    return state

## Step 3: Write the Routing Function

The routing function returns the name of the next node. The return value must be one of the keys in the edges map passed to `add_conditional_edges()`.

In [ ]:
def route_by_type(state: QAState) -> str:
    """Return the name of the next node based on the question type.

    The return value must be one of the keys in the edges map passed to
    add_conditional_edges(). If the value is unexpected, the graph raises.
    """
    if state.question_type == "factual":
        return "factual"
    elif state.question_type == "opinion":
        return "opinion"
    elif state.question_type == "creative":
        return "creative"
    else:
        # Anything unrecognized routes to the fallback handler
        return "unknown"

## Step 4: Build the Graph with Conditional Edges

After `classify` runs, call `route_by_type(state)`. The return value is looked up in the route map to find the next node.

In [ ]:
def build_graph():
    graph = StateGraph(QAState)

    graph.add_node("classify",        classify_question)
    graph.add_node("handle_factual",  handle_factual)
    graph.add_node("handle_opinion",  handle_opinion)
    graph.add_node("handle_creative", handle_creative)
    graph.add_node("handle_unknown",  handle_unknown)

    graph.set_entry_point("classify")

    # add_conditional_edges(source, routing_fn, route_map)
    # After `classify` runs, call route_by_type(state).
    # The return value is looked up in the route_map to find the next node.
    graph.add_conditional_edges(
        "classify",
        route_by_type,
        {
            "factual":  "handle_factual",
            "opinion":  "handle_opinion",
            "creative": "handle_creative",
            "unknown":  "handle_unknown",
        }
    )

    # All handlers are terminal nodes — no outgoing edges needed
    return graph.compile()

## Complete Working Example

Run four different questions through the graph — factual, opinion, creative, and nonsensical — to see all routes exercised.

In [ ]:
async def answer(question: str) -> str:
    compiled = build_graph()
    initial_state = QAState(question=question)
    final_state = await compiled.arun(initial_state)
    return final_state.answer


async def main():
    questions = [
        "What is the speed of light in a vacuum?",
        "Should companies prioritize profit over environmental sustainability?",
        "Write a short poem about a robot who dreams of being a gardener.",
        "Blorg florp snizzle wumph?",
    ]

    for q in questions:
        print(f"\nQ: {q}")
        answer_text = await answer(q)
        print(f"A: {answer_text[:200]}")

asyncio.run(main())

## Next Steps

- **[Fan-Out / Fan-In](fan-out-fan-in.ipynb)** — run multiple branches in parallel instead of choosing one
- **[Linear Workflow](linear-workflow.ipynb)** — start here if you haven't yet
- **[Human-in-the-Loop](human-in-the-loop.ipynb)** — pause the graph at a decision point and wait for human input